In [3]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent   
BOOK_DIR = BASE_DIR / "book"

books = pd.read_csv(BOOK_DIR / "books.csv")
book_tags = pd.read_csv(BOOK_DIR / "book_tags.csv")
tags = pd.read_csv(BOOK_DIR / "tags.csv")



In [4]:
books.columns

Index(['id', 'book_id', 'best_book_id', 'work_id', 'books_count', 'isbn',
       'isbn13', 'authors', 'original_publication_year', 'original_title',
       'title', 'language_code', 'average_rating', 'ratings_count',
       'work_ratings_count', 'work_text_reviews_count', 'ratings_1',
       'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5', 'image_url',
       'small_image_url'],
      dtype='str')

In [5]:
book_tags_merged = book_tags.merge(tags, on='tag_id')

top_tags = (book_tags_merged
    .sort_values('count', ascending=False)
    .groupby('goodreads_book_id')
    .head(5)
)

tags_per_book = (top_tags
    .groupby('goodreads_book_id')['tag_name']
    .apply(lambda x: ' '.join(x))
    .reset_index()
)

print(tags_per_book.shape) 

(10000, 2)


In [6]:
tags_per_book

,goodreads_book_id,tag_name
0,1,to-read fantasy favorites currently-reading yo...
1,2,to-read currently-reading fantasy favorites ch...
2,3,to-read favorites fantasy currently-reading yo...
3,5,favorites fantasy currently-reading young-adul...
4,6,fantasy young-adult fiction harry-potter owned
...,...,...
9995,31538647,to-read currently-reading fantasy short-storie...
9996,31845516,to-read currently-reading memoir non-fiction n...
9997,32075671,to-read young-adult favorites contemporary cur...
9998,32848471,to-read funny new-adult office-romance alpha-male


In [7]:
books = books.merge(
    tags_per_book, 
    left_on='book_id',
    right_on='goodreads_book_id', 
    how='left'
)

print(books.shape)

(10000, 25)


In [8]:
books.columns

Index(['id', 'book_id', 'best_book_id', 'work_id', 'books_count', 'isbn',
       'isbn13', 'authors', 'original_publication_year', 'original_title',
       'title', 'language_code', 'average_rating', 'ratings_count',
       'work_ratings_count', 'work_text_reviews_count', 'ratings_1',
       'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5', 'image_url',
       'small_image_url', 'goodreads_book_id', 'tag_name'],
      dtype='str')

In [9]:
books['content'] = (
    books['title'].fillna('') + ' ' + 
    books['authors'].fillna('') + ' ' + 
    books['tag_name'].fillna('')
)

books[['title', 'authors', 'tag_name', 'content']]

,title,authors,tag_name,content
0,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,favorites currently-reading young-adult fictio...,"The Hunger Games (The Hunger Games, #1) Suzann..."
1,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré",to-read favorites fantasy currently-reading yo...,Harry Potter and the Sorcerer's Stone (Harry P...
2,"Twilight (Twilight, #1)",Stephenie Meyer,young-adult fantasy favorites vampires ya,"Twilight (Twilight, #1) Stephenie Meyer young-..."
3,To Kill a Mockingbird,Harper Lee,classics favorites to-read classic historical-...,To Kill a Mockingbird Harper Lee classics favo...
4,The Great Gatsby,F. Scott Fitzgerald,classics favorites fiction classic books-i-own,The Great Gatsby F. Scott Fitzgerald classics ...
...,...,...,...,...
9995,"Bayou Moon (The Edge, #2)",Ilona Andrews,to-read urban-fantasy fantasy romance paranormal,"Bayou Moon (The Edge, #2) Ilona Andrews to-rea..."
9996,"Means of Ascent (The Years of Lyndon Johnson, #2)",Robert A. Caro,to-read biography history politics non-fiction,"Means of Ascent (The Years of Lyndon Johnson, ..."
9997,The Mauritius Command,Patrick O'Brian,to-read historical-fiction fiction historical ...,The Mauritius Command Patrick O'Brian to-read ...
9998,Cinderella Ate My Daughter: Dispatches from th...,Peggy Orenstein,to-read non-fiction nonfiction parenting feminism,Cinderella Ate My Daughter: Dispatches from th...


## Converting Embedding

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(
    books['content'].tolist(), 
    show_progress_bar=True,
    batch_size=64
)

print(embeddings.shape)  # (10000, 384)

c:\Users\bhand\OneDrive\Documents\PYTHON\Book recomend\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\bhand\OneDrive\Documents\PYTHON\Book recomend\myenv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bhand\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer M

(10000, 384)


In [ ]:
import numpy as np

# Embeddings save  (numpy array)
np.save('embeddings.npy', embeddings)

#model save to pickel file
books.to_pickle('books_processed.pkl')

print("Saved!")

Saved!
